In [ ]:
using Plots
using LinearAlgebra
using Random
using Statistics
plotly()

In [ ]:
# Normalised wavelength
λ = 1.0

# ULA parameters
N = 3
d = 0.5 * λ

# Antenna positions
x = (0:N-1) .* d

# True direction of arrival
theta_true_deg = 30.0
theta_true = deg2rad(theta_true_deg)

# Signal received at each antenna
received = exp.(-im .* (2π / λ) .* x .* sin(theta_true))

# Relative phases using A1 as the reference
measured_phase = angle.(received[1] .* conj.(received[2:end]))

measured_phase_deg = rad2deg.(measured_phase)

println("True AoA = ", theta_true_deg, "°")
println("Measured phases A1 - Ak = ", measured_phase_deg)

In [ ]:
# Candidate directions over the full azimuth
candidate_angles = 0.0:0.1:359.9

# Store RMSE for each candidate direction
RMSE_table = Float64[]

for angle_deg in candidate_angles

    theta = deg2rad(angle_deg)

    # Expected signal at each antenna for this candidate direction
    expected = exp.(-im .* (2π / λ) .* x .* sin(theta))

    # Expected A1 - Ak relative phases
    expected_phase = angle.(expected[1] .* conj.(expected[2:end]))

    # Wrapped phase error
    phase_error = angle.(exp.(im .* (measured_phase .- expected_phase)))

    # RMSE across the four antenna pairs
    rmse = sqrt(sum(phase_error.^2) / length(phase_error))

    # Store RMSE in degrees
    push!(RMSE_table, rad2deg(rmse))
end

# Find the minimum RMSE
minimum_RMSE = minimum(RMSE_table)

# Find all angles with essentially the same minimum
best_indices = findall(rmse -> rmse <= minimum_RMSE + 1e-6, RMSE_table)

best_angles = candidate_angles[best_indices]

println("Minimum RMSE = ", minimum_RMSE, "°")
println("Best matching angle(s) = ", best_angles)

In [ ]:
# ------------------------------------------------------------
# Find important local RMSE minima
# ------------------------------------------------------------

local_min_indices = Int[]

for i in 2:length(RMSE_table)-1

    if RMSE_table[i] < RMSE_table[i-1] &&
       RMSE_table[i] < RMSE_table[i+1]

        push!(local_min_indices, i)
    end
end

# Only keep minima very close to zero RMSE
# This identifies directions with essentially the same phase pattern
ambiguity_threshold = 1.0  # degrees

strong_indices = [
    i for i in local_min_indices
    if RMSE_table[i] <= ambiguity_threshold
]

strong_angles = candidate_angles[strong_indices]
strong_RMSE = RMSE_table[strong_indices]

println("Important RMSE minima:")

for i in eachindex(strong_angles)
    println(
        round(strong_angles[i], digits=1),
        "°  ->  RMSE = ",
        round(strong_RMSE[i], digits=4),
        "°"
    )
end


# ------------------------------------------------------------
# Plot RMSE direction search
# ------------------------------------------------------------

spacing = round(d / λ, digits=2)

p = plot(
    candidate_angles,
    RMSE_table,
    linewidth = 2,
    xlabel = "Candidate azimuth (degrees)",
    ylabel = "Wrapped phase RMSE (degrees)",
    title = "ULA Direction Search, d = $(spacing)λ",
    label = false,
    xlims = (0, 360),
    xticks = 0:45:360
)

scatter!(
    p,
    strong_angles,
    strong_RMSE,
    markersize = 6,
    label = "Ambiguous minimum"
)

display(p)


# ------------------------------------------------------------
# Save figure
# ------------------------------------------------------------

spacing_label = replace(string(spacing), "." => "p")

savefig(
    p,
    "ULA_RMSE_d_$(spacing_label)lambda.png"
)

In [ ]:
spacing_label = replace(string(round(d/λ, digits=2)), "." => "p")

savefig(
    p,
    "ULA_RMSE_d_$(spacing_label)lambda.png"
)